# Phase 2: CulturalPangea-7B + Qwen3.5-4B candidates for the ensembles (SemEval 2027 Task 6, Task 2)

Stages on the first 500 dev items per track: `pangea_p0` (reproduces the earlier Pangea run), `qwen_z1` (zero-shot control
for Phase 1), `cascade_qwen2pangea` (Pangea with Qwen's answer as a hint), `fusion_qwen` (Qwen fuses both answers), and
optionally two sampled Qwen candidates for MBR. Pangea runs in its own virtualenv (transformers 4.37), Qwen in the main
environment (transformers 5.x). Download `phase2_results.zip` from the Output tab.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, Internet **on**. Smoke run first (`LIMIT = 4`), then the full run.

## 1. Config

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-experiments.git"
BRANCH = "main"
LIMIT = 4              # smoke test: 4 dev items per track. Set to None for the full run.
BUDGET_HOURS = 11.0
SAMPLES = False        # True: two extra sampled Qwen candidates (about +1.5 h) for MBR
DEVTEST = False        # True: also answer devtest with pangea_p0 and qwen_z1 if time allows
PANGEA_WORKERS = None  # set to 2 after the smoke run if Pangea's peak_mem_gb is below ~7 GB
WORK = "/kaggle/working/results"

## 2. Code and dependencies (main = Qwen environment; the Pangea venv is built and verified by the orchestrator)

In [ ]:
import glob, os, subprocess, sys
CODE = "/kaggle/working/mmcqa-experiments"
if not os.path.isdir(CODE):
    local = glob.glob("/kaggle/input/**/mmcqa_exp/__init__.py", recursive=True)    # a Kaggle Dataset copy also works
    if local:
        subprocess.run(["cp", "-r", os.path.dirname(os.path.dirname(local[0])), CODE], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi -L
!pip install -q -r {CODE}/requirements/qwen.txt
if globals().get("INSTALL_FLA", False):        # optional fast kernels for Qwen3.5's linear-attention layers
    !pip install -q flash-linear-attention
    print(subprocess.run([sys.executable, "-c", "import fla; print('fla', fla.__version__)"], capture_output=True, text=True))
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|bitsandbytes|accelerate|huggingface-hub|tokenizers) "

In [ ]:
try:                                   # optional: a Kaggle secret HF_TOKEN avoids anonymous download limits
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: continuing with anonymous downloads")

## 3. Run the stages

In [ ]:
args = f"--workdir {WORK} --budget-hours {BUDGET_HOURS}"
if LIMIT: args += f" --limit {LIMIT}"
if SAMPLES: args += " --samples"
if DEVTEST: args += " --devtest"
if PANGEA_WORKERS: args += f" --pangea-workers {PANGEA_WORKERS}"
!cd {CODE} && python -m mmcqa_exp.phase2 {args}

## 4. What happened

In [ ]:
import json, glob
print(open(f"{WORK}/logs/phase2_completion.json").read()[:6000])
print(open(f"{WORK}/logs/phase2_run_config.json").read()[:2500])
for p in sorted(glob.glob(f"{WORK}/predictions/*/dev_arz.jsonl")):
    print(p); [print("  ", json.loads(l)["question"], "->", json.loads(l)["answer"]) for l in open(p).readlines()[:2]]

## 5. Package

In [ ]:
ZIPNAME = "phase2_results.zip"

import shutil, pathlib
media = pathlib.Path(os.environ.get("MMCQA_MEDIA", "/kaggle/temp/mmcqa_media"))
for tmp in (WORK + "/mmcultureqa", "/kaggle/working/mmcqa_media", "/kaggle/working/mmcqa_envs", str(media), str(media.parent / "mmcqa_envs")):
    shutil.rmtree(tmp, ignore_errors=True)
!cd /kaggle/working && zip -qr {ZIPNAME} results && ls -lh {ZIPNAME}